# HealthBot: AI-Powered Patient Education System

This notebook runs a LangGraph learning workflow. It uses Tavily results only, provides a patient-friendly summary with source citations, and resets all topic data before a new session.

In [ ]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.graph import START, END, StateGraph

load_dotenv('config.env', override=True)
assert os.getenv('GROQ_API_KEY'), 'Add GROQ_API_KEY to config.env'
assert os.getenv('TAVILY_API_KEY'), 'Add TAVILY_API_KEY to config.env'
print('Configuration loaded successfully.')

In [ ]:
llm = ChatGroq(model=os.getenv('GROQ_MODEL', 'openai/gpt-oss-20b'), temperature=0)
tavily_search = TavilySearch(max_results=4, search_depth='advanced')
print('Groq model and Tavily search tool configured successfully.')

In [ ]:
class HealthBotState(TypedDict, total=False):
    patient_topic: str
    raw_search_results: list[dict]
    patient_summary: str
    quiz_question: str
    patient_answer: str
    quiz_grade: str
    feedback_explanation: str
    continue_session: bool

print('HealthBotState created successfully.')

In [ ]:
def ask_topic(_: HealthBotState):
    return {'patient_topic': input('What health topic or medical condition would you like to learn about? ').strip()}

def search_medical_information(state: HealthBotState):
    query = (f"{state['patient_topic']} patient education site:cdc.gov OR site:nih.gov OR "
             'site:medlineplus.gov OR site:who.int')
    response = tavily_search.invoke(query)
    results = response.get('results', [])
    if not results:
        raise RuntimeError('No reputable medical sources were returned. Please try another topic.')
    return {'raw_search_results': results}

def source_material(results):
    return '\n\n'.join(
        f"SOURCE: {item.get('title', 'Untitled')}\nURL: {item.get('url', '')}\nCONTENT: {item.get('content', '')}"
        for item in results
    )

In [ ]:
def summarize_content(state: HealthBotState):
    prompt = f'''You are a patient education assistant. Write exactly 3 or 4 short, patient-friendly paragraphs about {state['patient_topic']}. Use ONLY the Tavily search material below. Do not use outside knowledge, diagnose, or give personalized medical advice. Every medical fact must have an inline citation using its source title in square brackets. Return only the summary.

TAVILY SEARCH MATERIAL:
{source_material(state['raw_search_results'])}'''
    return {'patient_summary': llm.invoke(prompt).content}

def present_summary(state: HealthBotState):
    print('\n' + '=' * 60 + '\nHEALTH INFORMATION\n' + '=' * 60)
    print(state['patient_summary'])
    print('\nSOURCES USED:')
    for item in state['raw_search_results']:
        print(f"- {item.get('title', 'Source')}: {item.get('url', '')}")
    input('\nPress Enter when you are ready for a comprehension check. ')
    return {}

In [ ]:
def generate_quiz_question(state: HealthBotState):
    prompt = f'''Create exactly ONE short, open-ended comprehension question using ONLY the summary below. It must be answerable from the summary alone. Do not reveal an answer, provide options, add commentary, or use outside knowledge. Return only the question.

HEALTH SUMMARY:
{state['patient_summary']}'''
    return {'quiz_question': llm.invoke(prompt).content.strip()}

def ask_quiz_answer(state: HealthBotState):
    print('\nCOMPREHENSION CHECK\n' + state['quiz_question'])
    return {'patient_answer': input('Your answer: ').strip()}

def grade_response(state: HealthBotState):
    prompt = f'''Grade the student's answer using ONLY the health summary below. Give exactly one letter grade (A, B, C, or D), then a brief justification. The justification must refer to one or more inline citations already present in the summary; do not invent sources or use outside information.

Use exactly this format:
Grade: <A/B/C/D>
Feedback: <brief justification with cited source title(s)>

HEALTH SUMMARY:
{state['patient_summary']}

QUESTION:
{state['quiz_question']}

STUDENT ANSWER:
{state['patient_answer']}'''
    feedback = llm.invoke(prompt).content.strip()
    return {'quiz_grade': feedback.splitlines()[0].replace('Grade:', '').strip(), 'feedback_explanation': feedback}

In [ ]:
def present_results(state: HealthBotState):
    print('\n' + '=' * 60 + '\nQUIZ RESULT\n' + '=' * 60)
    print(state['feedback_explanation'])
    return {}

def ask_another_topic(_: HealthBotState):
    answer = input('\nWould you like to learn about another topic? (yes/no): ').strip().lower()
    return {'continue_session': answer in {'y', 'yes'}}

def reset_topic(_: HealthBotState):
    # Clear every item containing the previous patient's health information.
    return {'patient_topic': '', 'raw_search_results': [], 'patient_summary': '',
            'quiz_question': '', 'patient_answer': '', 'quiz_grade': '',
            'feedback_explanation': '', 'continue_session': True}

def route_session(state: HealthBotState):
    return 'new_topic' if state.get('continue_session') else 'end'

In [ ]:
builder = StateGraph(HealthBotState)
for name, node in {
    'ask_topic': ask_topic, 'search_medical_information': search_medical_information,
    'summarize_content': summarize_content, 'present_summary': present_summary,
    'generate_quiz_question': generate_quiz_question, 'ask_quiz_answer': ask_quiz_answer,
    'grade_response': grade_response, 'present_results': present_results,
    'ask_another_topic': ask_another_topic, 'reset_topic': reset_topic,
}.items():
    builder.add_node(name, node)

builder.add_edge(START, 'ask_topic')
for source, target in [('ask_topic', 'search_medical_information'), ('search_medical_information', 'summarize_content'), ('summarize_content', 'present_summary'), ('present_summary', 'generate_quiz_question'), ('generate_quiz_question', 'ask_quiz_answer'), ('ask_quiz_answer', 'grade_response'), ('grade_response', 'present_results'), ('present_results', 'ask_another_topic'), ('reset_topic', 'ask_topic')]:
    builder.add_edge(source, target)
builder.add_conditional_edges('ask_another_topic', route_session, {'new_topic': 'reset_topic', 'end': END})
healthbot = builder.compile()
print('HealthBot LangGraph workflow compiled successfully.')

In [ ]:
# Run this cell to start the complete patient learning session.
healthbot.invoke({}, {'recursion_limit': 1000})